# GAT-v4 | Phase 7B — Baseline Comparison

**Companion to Phase 7 (GATv4 training).** Runs classical survival baselines on
the exact same 15 CV folds (`cv_folds_repeated_5x3.json`) and the exact same
fold-aware feature tensors produced in Phase 6, so results are directly
comparable to the GATv4 numbers (mean test C-index 0.7697 ± 0.0442, mean IBS
0.1615, mean iAUC 0.7913 — see `results/tables/cv_results.csv`).

### Why this notebook exists
A GNN result only means something in context. Reviewers will ask "how much
better than a plain Cox model?" — this notebook answers that with four
baselines of increasing complexity:

| # | Baseline | Features used | Captures graph structure? |
|---|----------|---------------|---------------------------|
| 1 | Clinical-only Cox-PH | age, stage, grade | No |
| 2 | Clinical + PCA-gene Cox-PH | clinical + top-k PCA of RNA z-scores | No |
| 3 | Random Survival Forest | clinical + PCA-gene features | No (non-linear, but no PPI edges) |
| 4 | Elastic-net Cox (CoxNet) | clinical + full gene panel (regularised) | No |

All four reuse the *same* per-fold train/val/test split, the *same*
fold-aware normalisation already baked into `feature_tensor.npy` /
`clinical_tensor.npy`, and the *same* evaluation metrics (C-index, IBS,
iAUC, log-rank) as Phase 7, by importing them directly rather than
re-implementing them.

### Requirements
```
pip install scikit-survival
```
`scikit-survival` provides `CoxPHSurvivalAnalysis`, `RandomSurvivalForest`,
and `CoxnetSurvivalAnalysis` with an API that mirrors scikit-learn.

---
| Cell | What |
|------|------|
| 1 | Imports & paths (reuses Phase 7 metric functions) |
| 2 | Metric functions ported from Phase 7 (C-index, IBS, iAUC, log-rank) |
| 3 | Data loader — flattens tensors to tabular form per fold |
| 4 | PCA gene-reduction helper (fold-aware, fit on train only) |
| 5 | Baseline 1 — Clinical-only Cox-PH |
| 6 | Baseline 2 — Clinical + PCA-gene Cox-PH |
| 7 | Baseline 3 — Random Survival Forest |
| 8 | Baseline 4 — Elastic-net Cox (CoxNet) on full gene panel |
| 9 | Aggregate all baselines across 15 folds |
| 10 | Comparison table: GATv4 vs. all baselines |
| 11 | Save results + audit YAML |

---
## Cell 1 — Imports & Paths

In [1]:
import sys, json, yaml, hashlib, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

try:
    from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
    from sksurv.ensemble import RandomSurvivalForest
    from sksurv.util import Surv
except ImportError:
    raise ImportError(
        "scikit-survival is required for this notebook.\n"
        "Install it with: pip install scikit-survival"
    )

# ── PATHS (same layout as Phase 7) ────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT = PROJECT_ROOT / "KIRC"
DATA_PROC = KIRC_ROOT / "data" / "processed"
DATA_SPLITS = KIRC_ROOT / "data" / "splits"
CKPT_DIR = KIRC_ROOT / "results" / "checkpointsibsLOG"   # GATv4 results (Phase 7)
BASELINE_DIR = KIRC_ROOT / "results" / "baselines"        # this notebook's outputs
TABLE_DIR = KIRC_ROOT / "results" / "tables"
AUDIT_DIR = KIRC_ROOT / "audit"
# ──────────────────────────────────────────────────────────────────────────

for d in [BASELINE_DIR, TABLE_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import MASTER_SEED, N_BOOTSTRAP

log = get_logger('phase7b_baselines')

# Number of PCA components for the gene-expression-reduced baselines.
# Kept small relative to n_train (~350) to avoid overfitting a linear Cox model.
N_PCA_COMPONENTS = 20

print("Imports OK.")
print(f"N_PCA_COMPONENTS = {N_PCA_COMPONENTS}")

2026-09-18 09:51:06 | INFO     | phase7b_baselines | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase7b_baselines_20260918_095106.log


Imports OK.
N_PCA_COMPONENTS = 20


---
## Cell 2 — Metric Functions (Ported from Phase 7)

These are copied verbatim from Phase 7 Cells 3/3B/3C/3D so this notebook has
no hidden dependency on that notebook's live kernel state, and can be run
standalone. Definitions, conventions (risk = higher-is-worse Cox linear
predictor), and train-only-fit leakage guards are identical.

In [2]:
def concordance_index(risks, times, events):
    """Harrell's C-index. risks: higher = worse prognosis."""
    risks = np.asarray(risks, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)

    event_idx = np.where(events == 1)[0]
    if len(event_idx) == 0:
        return 0.5

    t_i = times[event_idx][:, None]
    r_i = risks[event_idx][:, None]
    t_j = times[None, :]
    r_j = risks[None, :]

    comparable = t_j > t_i
    concordant = comparable & (r_i > r_j)
    discordant = comparable & (r_i < r_j)
    tied = comparable & (r_i == r_j)

    total = concordant.sum() + discordant.sum() + tied.sum()
    if total == 0:
        return 0.5
    return (concordant.sum() + 0.5 * tied.sum()) / total


def bootstrap_ci(risks, times, events, n_boot=N_BOOTSTRAP, seed=MASTER_SEED):
    rng = np.random.default_rng(seed)
    n = len(risks)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots.append(concordance_index(risks[idx], times[idx], events[idx]))
    boots = np.array(boots)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def breslow_baseline_hazard(risks, times, events):
    risks = np.asarray(risks, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    exp_risk = np.exp(risks)

    order = np.argsort(times)
    t_sorted = times[order]
    e_sorted = events[order]
    er_sorted = exp_risk[order]

    suffix_sum = np.cumsum(er_sorted[::-1])[::-1]
    uniq_event_t = np.unique(t_sorted[e_sorted == 1])
    if len(uniq_event_t) == 0:
        return np.array([0.0]), np.array([0.0])

    pos = np.searchsorted(t_sorted, uniq_event_t, side='left')
    denom = suffix_sum[pos]
    denom = np.where(denom > 0, denom, 1.0)
    d_k = np.array([e_sorted[t_sorted == t].sum() for t in uniq_event_t])
    cumhaz = np.cumsum(d_k / denom)
    return uniq_event_t, cumhaz


def predict_survival_function(risks, baseline_times, baseline_cumhaz, eval_times):
    risks = np.asarray(risks, dtype=np.float64)
    eval_times = np.asarray(eval_times, dtype=np.float64)
    idx = np.searchsorted(baseline_times, eval_times, side='right') - 1
    H0_at_eval = np.where(idx >= 0,
        baseline_cumhaz[np.clip(idx, 0, len(baseline_cumhaz) - 1)], 0.0)
    exp_risk = np.exp(risks)[:, None]
    return np.exp(-H0_at_eval[None, :] * exp_risk)


def fit_km_censoring(times, events):
    times = np.asarray(times, dtype=np.float64)
    cens_ind = 1.0 - np.asarray(events, dtype=np.float64)
    order = np.argsort(times)
    t_sorted = times[order]
    c_sorted = cens_ind[order]
    n = len(times)
    uniq_t = np.unique(t_sorted)
    cum_before = np.searchsorted(t_sorted, uniq_t, side='left')
    n_at_risk = n - cum_before
    d_cens = np.array([c_sorted[t_sorted == t].sum() for t in uniq_t])
    factor = 1.0 - d_cens / np.where(n_at_risk > 0, n_at_risk, 1.0)
    G_step = np.cumprod(factor)
    return uniq_t, G_step


def km_eval(query_times, uniq_t, surv, side='right'):
    query_times = np.asarray(query_times, dtype=np.float64)
    if len(query_times) == 0:
        return np.array([])
    if side == 'right':
        idx = np.searchsorted(uniq_t, query_times, side='right') - 1
    else:
        idx = np.searchsorted(uniq_t, query_times, side='left') - 1
    g = np.where(idx >= 0, surv[np.clip(idx, 0, len(surv) - 1)], 1.0)
    return np.clip(g, 1e-3, 1.0)


def brier_score_ipcw(eval_times, times, events, surv_probs, cens_uniq_t, cens_surv):
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    n = len(times)
    bs = np.zeros(len(eval_times), dtype=np.float64)
    for k, t in enumerate(eval_times):
        s_t = surv_probs[:, k]
        is_case = (times <= t) & (events == 1)
        is_control = (times > t)
        term = 0.0
        if is_case.any():
            g_case = km_eval(times[is_case], cens_uniq_t, cens_surv, side='left')
            term += (np.square(s_t[is_case]) / g_case).sum()
        if is_control.any():
            g_ctrl = km_eval(np.full(int(is_control.sum()), t), cens_uniq_t, cens_surv, side='right')
            term += (np.square(1.0 - s_t[is_control]) / g_ctrl).sum()
        bs[k] = term / n
    return bs


def integrated_brier_score(train_risks, train_times, train_events,
                            test_risks, test_times, test_events, n_time_points=100):
    train_risks = np.asarray(train_risks, dtype=np.float64)
    train_times = np.asarray(train_times, dtype=np.float64)
    train_events = np.asarray(train_events, dtype=np.float64)
    test_risks = np.asarray(test_risks, dtype=np.float64)
    test_times = np.asarray(test_times, dtype=np.float64)
    test_events = np.asarray(test_events, dtype=np.float64)

    baseline_t, baseline_H0 = breslow_baseline_hazard(train_risks, train_times, train_events)
    cens_uniq_t, cens_surv = fit_km_censoring(train_times, train_events)

    t_max = min(train_times.max(), test_times.max())
    positive_test_times = test_times[test_times > 0]
    t_min = max(positive_test_times.min(), 1e-3) if len(positive_test_times) else 1e-3
    eval_times = np.linspace(t_min, t_max, n_time_points)

    surv_probs = predict_survival_function(test_risks, baseline_t, baseline_H0, eval_times)
    bs_t = brier_score_ipcw(eval_times, test_times, test_events, surv_probs, cens_uniq_t, cens_surv)

    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    ibs = _trapz(bs_t, eval_times) / (eval_times[-1] - eval_times[0])
    return float(ibs), eval_times, bs_t


def time_dependent_auc(train_times, train_events, risks, times, events, eval_times):
    risks = np.asarray(risks, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.float64)
    eval_times = np.asarray(eval_times, dtype=np.float64)
    cens_uniq_t, cens_surv = fit_km_censoring(train_times, train_events)

    auc_t = np.full(len(eval_times), np.nan)
    for k, t in enumerate(eval_times):
        is_case = (times <= t) & (events == 1)
        is_ctrl = (times > t)
        n_ctrl = is_ctrl.sum()
        if is_case.sum() == 0 or n_ctrl == 0:
            continue
        g_case = km_eval(times[is_case], cens_uniq_t, cens_surv, side='left')
        w_case = 1.0 / g_case
        r_case = risks[is_case][:, None]
        r_ctrl = risks[is_ctrl][None, :]
        score = (r_case > r_ctrl).astype(np.float64) + 0.5 * (r_case == r_ctrl)
        numer = (w_case[:, None] * score).sum()
        denom = w_case.sum() * n_ctrl
        auc_t[k] = numer / denom if denom > 0 else np.nan
    return auc_t


def integrated_time_dependent_auc(train_times, train_events, risks, times, events, n_time_points=100):
    train_times = np.asarray(train_times, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)
    t_max = min(train_times.max(), times.max())
    positive_times = times[times > 0]
    t_min = max(positive_times.min(), 1e-3) if len(positive_times) else 1e-3
    eval_times = np.linspace(t_min, t_max, n_time_points)
    auc_curve = time_dependent_auc(train_times, train_events, risks, times, events, eval_times)
    valid = ~np.isnan(auc_curve)
    if valid.sum() < 2:
        return float('nan'), eval_times, auc_curve
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    span = eval_times[valid][-1] - eval_times[valid][0]
    iauc = _trapz(auc_curve[valid], eval_times[valid]) / span if span > 0 else float(auc_curve[valid][0])
    return float(iauc), eval_times, auc_curve


def logrank_test(times_a, events_a, times_b, events_b):
    times_a = np.asarray(times_a, dtype=np.float64)
    events_a = np.asarray(events_a, dtype=np.float64)
    times_b = np.asarray(times_b, dtype=np.float64)
    events_b = np.asarray(events_b, dtype=np.float64)
    times = np.concatenate([times_a, times_b])
    events = np.concatenate([events_a, events_b])
    group = np.concatenate([np.ones_like(times_a), np.zeros_like(times_b)])

    event_times = np.unique(times[events == 1])
    if len(event_times) == 0:
        return 0.0, 1.0, 0.0

    O1 = E1 = V = 0.0
    for t in event_times:
        at_risk = times >= t
        n_i = at_risk.sum()
        if n_i <= 1:
            continue
        n_1i = (at_risk & (group == 1)).sum()
        n_2i = n_i - n_1i
        deaths_here = at_risk & (times == t) & (events == 1)
        d_i = deaths_here.sum()
        d_1i = (deaths_here & (group == 1)).sum()
        e_1i = d_i * n_1i / n_i
        v_i = d_i * (n_1i / n_i) * (n_2i / n_i) * (n_i - d_i) / (n_i - 1)
        O1 += d_1i
        E1 += e_1i
        V += v_i

    if V == 0:
        return 0.0, 1.0, float(O1 - E1)
    chi2 = (O1 - E1) ** 2 / V
    p = float(1.0 - stats.chi2.cdf(chi2, df=1))
    return float(chi2), p, float(O1 - E1)


def median_split_groups(train_risks, risks):
    threshold = np.median(np.asarray(train_risks, dtype=np.float64))
    return np.asarray(risks, dtype=np.float64) > threshold


print("Metric functions ported from Phase 7 (C-index, IBS, iAUC, log-rank).")

Metric functions ported from Phase 7 (C-index, IBS, iAUC, log-rank).


---
## Cell 3 — Data Loader (Tabular Form)

Reads the exact same `feature_tensor.npy` / `clinical_tensor.npy` /
`survival_labels.npy` / `split_indices.json` produced by Phase 6, but
flattens the per-gene tensor to a plain `(n_samples, n_genes)` matrix using
only the RNA z-score channel (dim 0) — the classical baselines below don't
use graph structure, so only the expression values matter, not the CNV or
degree channels which exist to feed the GAT.

In [3]:
def load_fold_tabular(fold_key):
    """Load one fold and return a flat tabular dict — RNA z-scores only
    (feature dim 0), clinical features, survival labels, split indices.
    """
    fold_dir = DATA_SPLITS / fold_key

    feat = np.load(fold_dir / 'feature_tensor.npy')       # (n_samp, n_nodes, 3)
    clin = np.load(fold_dir / 'clinical_tensor.npy')      # (n_samp, 3)
    surv = np.load(fold_dir / 'survival_labels.npy')      # (n_samp, 2)

    rna = feat[:, :, 0]   # (n_samp, n_nodes) — VST z-score channel only

    with open(fold_dir / 'split_indices.json') as f:
        split_idx = json.load(f)

    times = surv[:, 0].astype(np.float64)
    events = surv[:, 1].astype(np.float64)

    return {
        'rna': rna, 'clin': clin, 'times': times, 'events': events,
        'train_idx': np.array(split_idx['train']),
        'val_idx':   np.array(split_idx['val']),
        'test_idx':  np.array(split_idx['test']),
    }


with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
    cv_splits = json.load(f)

_check = load_fold_tabular(list(cv_splits.keys())[0])
print(f"RNA matrix       : {_check['rna'].shape}")
print(f"Clinical matrix  : {_check['clin'].shape}")
print(f"Train/val/test   : {len(_check['train_idx'])}/{len(_check['val_idx'])}/{len(_check['test_idx'])}")
print("Data loader defined.")

RNA matrix       : (500, 4414)
Clinical matrix  : (500, 3)
Train/val/test   : 361/40/99
Data loader defined.


---
## Cell 4 — PCA Gene-Reduction Helper

Fold-aware: PCA is fit on the **train** split's RNA z-scores only, then
applied to val/test — same leakage-safe convention used throughout Phase 6/7.
Reducing ~4,414 genes to `N_PCA_COMPONENTS` components keeps the linear
Cox model well under the `n_features < n_train` regime it needs to converge
(train sets have ~350 patients).

In [4]:
def pca_reduce(data, n_components=N_PCA_COMPONENTS, seed=MASTER_SEED):
    """Fit PCA on train-fold RNA z-scores, transform all samples.
    Returns (n_samples, n_components) array.
    """
    train_rna = data['rna'][data['train_idx']]
    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(train_rna)
    reduced = pca.transform(data['rna'])
    return reduced, pca


def build_design_matrix(data, use_pca_genes=False, n_components=N_PCA_COMPONENTS):
    """clinical-only or clinical+PCA-gene design matrix, all samples."""
    X = data['clin'].copy()
    if use_pca_genes:
        gene_feats, _ = pca_reduce(data, n_components=n_components)
        X = np.concatenate([X, gene_feats], axis=1)
    return X


print("PCA gene-reduction helper defined (fit on TRAIN split only).")

PCA gene-reduction helper defined (fit on TRAIN split only).


---
## Cell 5 — Baseline 1: Clinical-Only Cox-PH

The floor every genomic model must beat: a linear Cox model using only
age, stage, and grade — no expression data at all. If GATv4 doesn't clear
this by a meaningful margin, the graph/expression signal isn't adding
value.

In [5]:
def run_coxph_baseline(cv_splits, use_pca_genes, n_components, baseline_name, alpha=0.1):
    """Generic Cox-PH baseline runner (clinical-only or clinical+PCA-gene)."""
    fold_results = []
    for fold_key, fold_def in cv_splits.items():
        data = load_fold_tabular(fold_key)
        train_idx, val_idx, test_idx = data['train_idx'], data['val_idx'], data['test_idx']

        X = build_design_matrix(data, use_pca_genes=use_pca_genes, n_components=n_components)
        y_struct = Surv.from_arrays(event=data['events'].astype(bool), time=data['times'])

        model = CoxPHSurvivalAnalysis(alpha=alpha, ties='breslow')
        model.fit(X[train_idx], y_struct[train_idx])

        # sksurv's predict() returns the risk score (linear predictor), higher = worse —
        # same convention as GATv4's `risk` output, so all metric functions apply unchanged.
        train_risks = model.predict(X[train_idx])
        val_risks = model.predict(X[val_idx])
        test_risks = model.predict(X[test_idx])

        test_times, test_events = data['times'][test_idx], data['events'][test_idx]
        train_times, train_events = data['times'][train_idx], data['events'][train_idx]

        test_cindex = concordance_index(test_risks, test_times, test_events)
        ci_lo, ci_hi = bootstrap_ci(test_risks, test_times, test_events)
        fold_ibs, _, _ = integrated_brier_score(train_risks, train_times, train_events,
                                                  test_risks, test_times, test_events)
        fold_iauc, _, _ = integrated_time_dependent_auc(train_times, train_events,
                                                          test_risks, test_times, test_events)
        hi_mask = median_split_groups(train_risks, test_risks)
        logrank_chi2, logrank_p, _ = logrank_test(
            test_times[hi_mask], test_events[hi_mask],
            test_times[~hi_mask], test_events[~hi_mask])

        fold_results.append({
            'baseline': baseline_name, 'fold_key': fold_key,
            'seed': fold_def['seed'], 'fold': fold_def['fold'],
            'test_cindex': round(float(test_cindex), 4),
            'test_ci_lo': round(ci_lo, 4), 'test_ci_hi': round(ci_hi, 4),
            'test_ibs': round(float(fold_ibs), 4),
            'test_iauc': round(float(fold_iauc), 4),
            'test_logrank_p': round(float(logrank_p), 6),
            'test_logrank_significant': bool(logrank_p < 0.05),
        })
    return pd.DataFrame(fold_results)


t0 = time.time()
clinical_only_df = run_coxph_baseline(cv_splits, use_pca_genes=False, n_components=0,
                                       baseline_name='Clinical-only Cox-PH')
print(f"Baseline 1 (Clinical-only Cox-PH) done in {time.time()-t0:.1f}s")
print(clinical_only_df[['fold_key', 'test_cindex', 'test_ibs', 'test_iauc']].to_string(index=False))
print(f"\nMean C-index: {clinical_only_df['test_cindex'].mean():.4f} ± {clinical_only_df['test_cindex'].std():.4f}")

Baseline 1 (Clinical-only Cox-PH) done in 3.4s
     fold_key  test_cindex  test_ibs  test_iauc
 seed42_fold0       0.7653    0.2432     0.6990
 seed42_fold1       0.7672    0.1685     0.6972
 seed42_fold2       0.8212    0.1255     0.8626
 seed42_fold3       0.6974    0.1684     0.7762
 seed42_fold4       0.8111    0.1736     0.7948
seed123_fold0       0.7436    0.1921     0.6873
seed123_fold1       0.7797    0.1789     0.7670
seed123_fold2       0.7839    0.2048     0.7263
seed123_fold3       0.7720    0.1635     0.7679
seed123_fold4       0.8329    0.1293     0.8644
seed456_fold0       0.7541    0.1776     0.8095
seed456_fold1       0.7762    0.1890     0.7860
seed456_fold2       0.8168    0.1499     0.6610
seed456_fold3       0.7721    0.1584     0.7646
seed456_fold4       0.7628    0.2143     0.7104

Mean C-index: 0.7771 ± 0.0342


---
## Cell 6 — Baseline 2: Clinical + PCA-Gene Cox-PH

Adds `N_PCA_COMPONENTS` principal components of the RNA expression panel to
the clinical features. This isolates how much of GATv4's edge comes from
"having gene expression at all" versus specifically from the **graph
structure and attention mechanism** — if this baseline nearly matches GATv4,
the graph isn't earning its complexity; if it falls well short, that's
direct evidence the PPI-graph attention is doing real work.

In [6]:
t0 = time.time()
clinical_pca_df = run_coxph_baseline(cv_splits, use_pca_genes=True, n_components=N_PCA_COMPONENTS,
                                      baseline_name='Clinical + PCA-gene Cox-PH')
print(f"Baseline 2 (Clinical + PCA-gene Cox-PH) done in {time.time()-t0:.1f}s")
print(clinical_pca_df[['fold_key', 'test_cindex', 'test_ibs', 'test_iauc']].to_string(index=False))
print(f"\nMean C-index: {clinical_pca_df['test_cindex'].mean():.4f} ± {clinical_pca_df['test_cindex'].std():.4f}")

Baseline 2 (Clinical + PCA-gene Cox-PH) done in 8.2s
     fold_key  test_cindex  test_ibs  test_iauc
 seed42_fold0       0.7902    0.2141     0.7481
 seed42_fold1       0.8022    0.1359     0.8162
 seed42_fold2       0.7617    0.1373     0.8154
 seed42_fold3       0.7168    0.1581     0.8077
 seed42_fold4       0.8201    0.1547     0.8164
seed123_fold0       0.7460    0.1860     0.7112
seed123_fold1       0.7589    0.1474     0.8214
seed123_fold2       0.7981    0.1864     0.7726
seed123_fold3       0.8134    0.1340     0.8380
seed123_fold4       0.8316    0.1100     0.8796
seed456_fold0       0.7622    0.1590     0.8360
seed456_fold1       0.7856    0.1808     0.8142
seed456_fold2       0.8292    0.1268     0.7592
seed456_fold3       0.7716    0.1397     0.8154
seed456_fold4       0.7640    0.1687     0.7589

Mean C-index: 0.7834 ± 0.0329


---
## Cell 7 — Baseline 3: Random Survival Forest

A non-linear baseline on the same clinical + PCA-gene features. RSF can
capture interactions and non-linear risk relationships that a linear Cox
model cannot, but still has no access to PPI topology — this checks whether
GATv4's advantage (if any) is really about the *graph*, not just about
non-linearity in general.

In [7]:
def run_rsf_baseline(cv_splits, n_components=N_PCA_COMPONENTS,
                     n_estimators=200, max_depth=5, seed=MASTER_SEED):
    fold_results = []
    for fold_key, fold_def in cv_splits.items():
        data = load_fold_tabular(fold_key)
        train_idx, val_idx, test_idx = data['train_idx'], data['val_idx'], data['test_idx']

        X = build_design_matrix(data, use_pca_genes=True, n_components=n_components)
        y_struct = Surv.from_arrays(event=data['events'].astype(bool), time=data['times'])

        rsf = RandomSurvivalForest(
            n_estimators=n_estimators, max_depth=max_depth,
            min_samples_leaf=10, n_jobs=-1, random_state=seed)
        rsf.fit(X[train_idx], y_struct[train_idx])

        train_risks = rsf.predict(X[train_idx])
        test_risks = rsf.predict(X[test_idx])

        test_times, test_events = data['times'][test_idx], data['events'][test_idx]
        train_times, train_events = data['times'][train_idx], data['events'][train_idx]

        test_cindex = concordance_index(test_risks, test_times, test_events)
        ci_lo, ci_hi = bootstrap_ci(test_risks, test_times, test_events)
        fold_ibs, _, _ = integrated_brier_score(train_risks, train_times, train_events,
                                                  test_risks, test_times, test_events)
        fold_iauc, _, _ = integrated_time_dependent_auc(train_times, train_events,
                                                          test_risks, test_times, test_events)
        hi_mask = median_split_groups(train_risks, test_risks)
        logrank_chi2, logrank_p, _ = logrank_test(
            test_times[hi_mask], test_events[hi_mask],
            test_times[~hi_mask], test_events[~hi_mask])

        fold_results.append({
            'baseline': 'Random Survival Forest', 'fold_key': fold_key,
            'seed': fold_def['seed'], 'fold': fold_def['fold'],
            'test_cindex': round(float(test_cindex), 4),
            'test_ci_lo': round(ci_lo, 4), 'test_ci_hi': round(ci_hi, 4),
            'test_ibs': round(float(fold_ibs), 4),
            'test_iauc': round(float(fold_iauc), 4),
            'test_logrank_p': round(float(logrank_p), 6),
            'test_logrank_significant': bool(logrank_p < 0.05),
        })
    return pd.DataFrame(fold_results)


t0 = time.time()
rsf_df = run_rsf_baseline(cv_splits)
print(f"Baseline 3 (Random Survival Forest) done in {time.time()-t0:.1f}s")
print(rsf_df[['fold_key', 'test_cindex', 'test_ibs', 'test_iauc']].to_string(index=False))
print(f"\nMean C-index: {rsf_df['test_cindex'].mean():.4f} ± {rsf_df['test_cindex'].std():.4f}")

Baseline 3 (Random Survival Forest) done in 14.2s
     fold_key  test_cindex  test_ibs  test_iauc
 seed42_fold0       0.7973    0.3264     0.7518
 seed42_fold1       0.7892    0.2536     0.7703
 seed42_fold2       0.7855    0.2455     0.8510
 seed42_fold3       0.6860    0.2772     0.7964
 seed42_fold4       0.8041    0.2482     0.8378
seed123_fold0       0.7605    0.2654     0.7355
seed123_fold1       0.7497    0.2637     0.8047
seed123_fold2       0.7697    0.2908     0.7756
seed123_fold3       0.8112    0.2502     0.8212
seed123_fold4       0.8256    0.2250     0.8738
seed456_fold0       0.7632    0.3017     0.8227
seed456_fold1       0.7891    0.2794     0.8058
seed456_fold2       0.8375    0.2527     0.7591
seed456_fold3       0.7346    0.2788     0.7618
seed456_fold4       0.7474    0.2379     0.7313

Mean C-index: 0.7767 ± 0.0387


---
## Cell 8 — Baseline 4: Elastic-Net Cox (CoxNet) on Full Gene Panel

Unlike baselines 1–2, this uses the **full** ~4,414-gene panel directly
(no PCA compression), relying on elastic-net regularisation (L1+L2) to
handle `n_features >> n_train`. This is the standard high-dimensional
genomics baseline (e.g. as used in TCGA pan-cancer survival papers) and the
fairest comparison for "does the graph structure add anything beyond raw
gene-level regularised linear risk scores?" `alpha_min_ratio` and `l1_ratio`
follow common defaults from the `scikit-survival` CoxNet examples.

In [8]:
def run_coxnet_baseline(cv_splits, l1_ratio=0.5, alpha_min_ratio=0.05, n_alphas=20):
    fold_results = []
    for fold_key, fold_def in cv_splits.items():
        data = load_fold_tabular(fold_key)
        train_idx, val_idx, test_idx = data['train_idx'], data['val_idx'], data['test_idx']

        # Full gene panel + clinical, standardised gene block (train stats already
        # z-scored per Phase 6, clinical already scaled) -> no extra scaling needed.
        X = np.concatenate([data['clin'], data['rna']], axis=1)
        y_struct = Surv.from_arrays(event=data['events'].astype(bool), time=data['times'])

        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio, alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas, max_iter=20000, fit_baseline_model=True)
        model.fit(X[train_idx], y_struct[train_idx])

        # Use the smallest-alpha (least regularised) fitted model on the path —
        # selected via validation C-index, same model-selection role as GATv4's
        # early-stopping-on-val-C-index in Phase 7.
        best_alpha_idx, best_val_ci = None, -1.0
        val_times, val_events = data['times'][val_idx], data['events'][val_idx]
        for i, alpha in enumerate(model.alphas_):
            risks_i = model.predict(X[val_idx], alpha=alpha)
            ci_i = concordance_index(risks_i, val_times, val_events)
            if ci_i > best_val_ci:
                best_val_ci, best_alpha_idx = ci_i, i
        best_alpha = model.alphas_[best_alpha_idx]

        train_risks = model.predict(X[train_idx], alpha=best_alpha)
        test_risks = model.predict(X[test_idx], alpha=best_alpha)

        test_times, test_events = data['times'][test_idx], data['events'][test_idx]
        train_times, train_events = data['times'][train_idx], data['events'][train_idx]

        test_cindex = concordance_index(test_risks, test_times, test_events)
        ci_lo, ci_hi = bootstrap_ci(test_risks, test_times, test_events)
        fold_ibs, _, _ = integrated_brier_score(train_risks, train_times, train_events,
                                                  test_risks, test_times, test_events)
        fold_iauc, _, _ = integrated_time_dependent_auc(train_times, train_events,
                                                          test_risks, test_times, test_events)
        hi_mask = median_split_groups(train_risks, test_risks)
        logrank_chi2, logrank_p, _ = logrank_test(
            test_times[hi_mask], test_events[hi_mask],
            test_times[~hi_mask], test_events[~hi_mask])

        n_nonzero = int(np.sum(model.coef_[:, best_alpha_idx] != 0))

        fold_results.append({
            'baseline': 'Elastic-net Cox (CoxNet)', 'fold_key': fold_key,
            'seed': fold_def['seed'], 'fold': fold_def['fold'],
            'test_cindex': round(float(test_cindex), 4),
            'test_ci_lo': round(ci_lo, 4), 'test_ci_hi': round(ci_hi, 4),
            'test_ibs': round(float(fold_ibs), 4),
            'test_iauc': round(float(fold_iauc), 4),
            'test_logrank_p': round(float(logrank_p), 6),
            'test_logrank_significant': bool(logrank_p < 0.05),
            'best_alpha': float(best_alpha),
            'n_nonzero_coefs': n_nonzero,
        })
    return pd.DataFrame(fold_results)


t0 = time.time()
coxnet_df = run_coxnet_baseline(cv_splits)
print(f"Baseline 4 (Elastic-net CoxNet) done in {time.time()-t0:.1f}s")
print(coxnet_df[['fold_key', 'test_cindex', 'test_ibs', 'test_iauc', 'n_nonzero_coefs']].to_string(index=False))
print(f"\nMean C-index: {coxnet_df['test_cindex'].mean():.4f} ± {coxnet_df['test_cindex'].std():.4f}")
print(f"Mean nonzero coefficients selected: {coxnet_df['n_nonzero_coefs'].mean():.0f}")

Baseline 4 (Elastic-net CoxNet) done in 8.8s
     fold_key  test_cindex  test_ibs  test_iauc  n_nonzero_coefs
 seed42_fold0       0.7511    0.1834     0.7883              145
 seed42_fold1       0.7016    0.1735     0.7292                8
 seed42_fold2       0.6715    0.1661     0.7277              133
 seed42_fold3       0.6922    0.1614     0.7703               72
 seed42_fold4       0.7171    0.2121     0.7445              126
seed123_fold0       0.6610    0.1924     0.7134              189
seed123_fold1       0.7492    0.1417     0.8432               70
seed123_fold2       0.6690    0.1939     0.7082               56
seed123_fold3       0.7789    0.1503     0.8285              194
seed123_fold4       0.6671    0.1982     0.6847                4
seed456_fold0       0.7062    0.1603     0.8218              183
seed456_fold1       0.6726    0.2214     0.6924                6
seed456_fold2       0.7267    0.1448     0.7541                7
seed456_fold3       0.7014    0.1716     0.72

---
## Cell 9 — Aggregate All Baselines Across 15 Folds

Same aggregation convention as Phase 7 Cell 7: per-baseline mean ± std,
range across folds, and log-rank significance rate.

In [9]:
all_baselines_df = pd.concat(
    [clinical_only_df, clinical_pca_df, rsf_df, coxnet_df],
    ignore_index=True
)

summary_rows = []
for name, grp in all_baselines_df.groupby('baseline'):
    summary_rows.append({
        'baseline': name,
        'mean_cindex': round(grp['test_cindex'].mean(), 4),
        'std_cindex': round(grp['test_cindex'].std(), 4),
        'min_cindex': round(grp['test_cindex'].min(), 4),
        'max_cindex': round(grp['test_cindex'].max(), 4),
        'mean_ibs': round(grp['test_ibs'].mean(), 4),
        'std_ibs': round(grp['test_ibs'].std(), 4),
        'mean_iauc': round(grp['test_iauc'].mean(), 4),
        'std_iauc': round(grp['test_iauc'].std(), 4),
        'frac_logrank_significant': round(grp['test_logrank_significant'].mean(), 3),
    })

baseline_summary_df = pd.DataFrame(summary_rows).sort_values('mean_cindex', ascending=False)
print(baseline_summary_df.to_string(index=False))

                  baseline  mean_cindex  std_cindex  min_cindex  max_cindex  mean_ibs  std_ibs  mean_iauc  std_iauc  frac_logrank_significant
Clinical + PCA-gene Cox-PH       0.7834      0.0329      0.7168      0.8316    0.1559   0.0274     0.8007    0.0427                       1.0
      Clinical-only Cox-PH       0.7771      0.0342      0.6974      0.8329    0.1758   0.0308     0.7583    0.0612                       1.0
    Random Survival Forest       0.7767      0.0387      0.6860      0.8375    0.2664   0.0265     0.7933    0.0428                       1.0
  Elastic-net Cox (CoxNet)       0.7048      0.0350      0.6610      0.7789    0.1758   0.0239     0.7508    0.0497                       1.0


---
## Cell 10 — GATv4 vs. All Baselines

Pulls GATv4's own aggregated results from Phase 7's `cv_results.csv` (must
have been run already) and appends it to the same comparison table — the
single table you'd put in a paper's "Comparison to baseline methods"
section.

In [10]:
gatv4_results_path = TABLE_DIR / 'cv_results.csv'
if gatv4_results_path.exists():
    gatv4_df = pd.read_csv(gatv4_results_path)
    gatv4_row = {
        'baseline': 'GATv4 (proposed)',
        'mean_cindex': round(gatv4_df['test_cindex'].mean(), 4),
        'std_cindex': round(gatv4_df['test_cindex'].std(), 4),
        'min_cindex': round(gatv4_df['test_cindex'].min(), 4),
        'max_cindex': round(gatv4_df['test_cindex'].max(), 4),
        'mean_ibs': round(gatv4_df['test_ibs'].mean(), 4),
        'std_ibs': round(gatv4_df['test_ibs'].std(), 4),
        'mean_iauc': round(gatv4_df['test_iauc'].mean(), 4),
        'std_iauc': round(gatv4_df['test_iauc'].std(), 4),
        'frac_logrank_significant': round(gatv4_df['test_logrank_significant'].mean(), 3)
            if 'test_logrank_significant' in gatv4_df.columns else np.nan,
    }
    final_comparison_df = pd.concat(
        [pd.DataFrame([gatv4_row]), baseline_summary_df], ignore_index=True
    ).sort_values('mean_cindex', ascending=False).reset_index(drop=True)
else:
    print("WARNING: Phase 7 cv_results.csv not found — run Phase 7 first to include GATv4 in the comparison.")
    final_comparison_df = baseline_summary_df.reset_index(drop=True)

print("=" * 70)
print("FINAL COMPARISON — GATv4 vs. Classical Survival Baselines")
print("=" * 70)
print(final_comparison_df.to_string(index=False))

if gatv4_results_path.exists():
    best_baseline_ci = baseline_summary_df['mean_cindex'].max()
    gatv4_ci = gatv4_row['mean_cindex']
    delta = gatv4_ci - best_baseline_ci
    print(f"\nGATv4 mean C-index      : {gatv4_ci:.4f}")
    print(f"Best baseline C-index   : {best_baseline_ci:.4f} ({baseline_summary_df.iloc[baseline_summary_df['mean_cindex'].argmax()]['baseline']})")
    print(f"Delta                   : {delta:+.4f}")
    if delta > 0.02:
        print("GATv4 shows a meaningful improvement over the best classical baseline.")
    elif delta > 0:
        print("GATv4 edges out the best baseline, but the margin is modest — consider")
        print("reporting effect size / CI overlap explicitly in the manuscript.")
    else:
        print("WARNING: best classical baseline is NOT beaten by GATv4 on mean C-index.")
        print("Re-examine model capacity, PPI graph quality, or feature engineering.")

FINAL COMPARISON — GATv4 vs. Classical Survival Baselines
                  baseline  mean_cindex  std_cindex  min_cindex  max_cindex  mean_ibs  std_ibs  mean_iauc  std_iauc  frac_logrank_significant
Clinical + PCA-gene Cox-PH       0.7834      0.0329      0.7168      0.8316    0.1559   0.0274     0.8007    0.0427                       1.0
      Clinical-only Cox-PH       0.7771      0.0342      0.6974      0.8329    0.1758   0.0308     0.7583    0.0612                       1.0
    Random Survival Forest       0.7767      0.0387      0.6860      0.8375    0.2664   0.0265     0.7933    0.0428                       1.0
          GATv4 (proposed)       0.7697      0.0458      0.6323      0.8170    0.1615   0.0332     0.7913    0.0494                       1.0
  Elastic-net Cox (CoxNet)       0.7048      0.0350      0.6610      0.7789    0.1758   0.0239     0.7508    0.0497                       1.0

GATv4 mean C-index      : 0.7697
Best baseline C-index   : 0.7834 (Clinical + PCA-gene Co

---
## Cell 11 — Save Results + Audit YAML

In [ ]:
all_baselines_path = TABLE_DIR / 'baseline_results_per_fold.csv'
all_baselines_df.to_csv(all_baselines_path, index=False)
print(f"Per-fold baseline results -> {all_baselines_path}")

summary_path = TABLE_DIR / 'baseline_vs_gatv4_summary.csv'
final_comparison_df.to_csv(summary_path, index=False)
print(f"Summary comparison table -> {summary_path}")

audit = {
    'phase': 'Phase 7B — Baseline Comparison',
    'timestamp': datetime.now().isoformat(),

    'baselines_run': {
        'clinical_only_coxph': {
            'features': 'age, stage, grade',
            'mean_cindex': float(clinical_only_df['test_cindex'].mean()),
        },
        'clinical_pca_coxph': {
            'features': f'clinical + top-{N_PCA_COMPONENTS} PCA of RNA z-scores',
            'mean_cindex': float(clinical_pca_df['test_cindex'].mean()),
        },
        'random_survival_forest': {
            'features': f'clinical + top-{N_PCA_COMPONENTS} PCA of RNA z-scores',
            'mean_cindex': float(rsf_df['test_cindex'].mean()),
        },
        'elastic_net_coxnet': {
            'features': 'clinical + full ~4,414-gene panel (regularised)',
            'mean_cindex': float(coxnet_df['test_cindex'].mean()),
            'mean_nonzero_coefs': float(coxnet_df['n_nonzero_coefs'].mean()),
        },
    },

    'comparison_to_gatv4': (
        final_comparison_df.to_dict(orient='records') if gatv4_results_path.exists()
        else 'GATv4 results not available at audit time — run Phase 7 first.'
    ),

    'leakage_guards': [
        'PCA fit on TRAIN split only per fold, applied to val/test',
        'Cox-PH / CoxNet fit on TRAIN split only per fold',
        'RSF fit on TRAIN split only per fold',
        'Baseline hazard (IBS) and censoring KM fit on TRAIN split only',
        'Median-split threshold (log-rank) fit on TRAIN split only',
        'CoxNet alpha selected via VAL C-index, never TEST',
    ],

    'next_phase': 'Phase 8 — Pathway Analysis + Biomarker Discovery',
    'status': 'COMPLETE',
}

audit_path = AUDIT_DIR / 'phase7b_baselines.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 7B COMPLETE")
print("=" * 55)
print(f"  Baselines run     : 4 (Clinical Cox-PH, Clinical+PCA Cox-PH, RSF, CoxNet)")
print(f"  Folds per baseline: {len(cv_splits)}")
print(f"  Best baseline     : {baseline_summary_df.iloc[0]['baseline']} "
      f"(C-index {baseline_summary_df.iloc[0]['mean_cindex']:.4f})")
if gatv4_results_path.exists():
    print(f"  GATv4 C-index     : {gatv4_row['mean_cindex']:.4f}")
print()
print("Next: pathway analysis + integrated gradients biomarker discovery")